# Salt dilution discharge

Historical analysis; scientific assumptions require review.

Source: `rjost1/FluvialSeismology_Analysis`, `Dilution/20250724_solution_injection_analysis.ipynb`, commit `fce22815c02abd61bebdff692fd49c128feb5f3a`. Calculations are preserved; imports and paths have been adapted. Stored source outputs have been cleared.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter inside the mora-fluvial-analysis repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import method_path, result_path
METHOD_ROOT = PROJECT_ROOT / 'salt_dilution'
os.chdir(METHOD_ROOT)


In [ ]:
from salt_dilution.scripts.processing import dilution_calculations as dil

In [ ]:
config_file = "config/2025-07-24_trial_01.ini"
files, injection, calibration = dil.load_config(config_file)

In [ ]:
injection_data = dil.get_file_paths(files['data_folder'],
                                    files['search_string'])
injection_data

In [ ]:
trial_data = dil.extract_fields(dil.read_csv(injection_data[0]))
cal_data = dil.extract_fields(dil.read_csv(files['calibration_file']))

In [ ]:
#Vadd - this variable declaration is unecessary. Just put the cal_data in the function signature
RC_ss = dil.secondary_solution_RC(injection_solution=int(calibration['volume_injection_solution sample']),
                                  Vo=int(calibration['initial_volume_secondary']))
RC = dil.calibration_RC(RC_sec=RC_ss,
                        additions_of_injection=cal_data['Additions_of_secondary_ml'],
                        Vc=int(cal_data['Calibration_volume_ml'][0]))

In [ ]:
slope_k = dil.calculate_k(RC_values=RC,EC_values=cal_data['Conductivity'])
print(f"Relative Concentration of Secondary Solution (ml/ml):{RC_ss}\nRelative Concentration after each addition (ml/ml):\n{RC}\nslope of the best-fit line (cm/uS): {slope_k}")

In [ ]:
dil.plot_calibration(cal_data=cal_data, RC=RC)

from matplotlib import pyplot as plt
plt.gcf().savefig(result_path("salt_dilution", "figures/calibration.png"), dpi=150, bbox_inches="tight")


In [ ]:
# get the time variables background, injection, total
bg_sec = dil.time_diff_sec(start=injection['bg_start'],end=injection['bg_end'])
trial_sec = dil.time_diff_sec(start=injection['bg_end'],end=injection['trial_end'])
total_sec = dil.time_diff_sec(start=injection['bg_start'],end=injection['record_end'])


In [ ]:
bg_value = dil.get_background_ec(start_time=0, end_time=bg_sec,ec_data=trial_data['Actual Conductivity (µS/cm)'])

In [ ]:
diff_sec, cumm_sec = dil.compute_time_differences_in_seconds(
    trial_data['Date Time'])

In [ ]:
Q = dil.calculate_Q(
    ec_bg=bg_value,
    ec_data=trial_data['Actual Conductivity (µS/cm)'],
    slope=slope_k,
    bg_sec=230,
    trial_sec=500,
    time_step=diff_sec,
    injection_volume=injection['injection_volume']
)

In [ ]:
Q

In [ ]:
dil.plot_injection(data=trial_data['Actual Conductivity (µS/cm)'], time_elapsed=cumm_sec, bg_sec=230,trial_sec=500,total_diff=diff_sec,show_background=True)

from matplotlib import pyplot as plt
plt.gcf().savefig(result_path("salt_dilution", "figures/conductivity_breakthrough_curve.png"), dpi=150, bbox_inches="tight")


In [ ]:
dil.plot_data(trial_data['Actual Conductivity (µS/cm)'], cumm_sec)

from matplotlib import pyplot as plt
plt.gcf().savefig(result_path("salt_dilution", "figures/conductivity_record.png"), dpi=150, bbox_inches="tight")


## Save this run

The table below records the preserved historical calculation. Successful execution does not establish scientific validity; review the limitations in the method README before interpreting the discharge estimate.

In [ ]:
import csv
import json
import shutil

with open(result_path("salt_dilution", "tables/discharge.csv"), "w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["discharge_m3_s", "calibration_k", "background_conductivity_us_cm"])
    writer.writerow([float(Q), float(slope_k), float(bg_value)])

config_snapshot = result_path("salt_dilution", "config.ini")
shutil.copyfile(config_file, config_snapshot)
info_path = Path(result_path("salt_dilution", "run_info.json"))
info = json.loads(info_path.read_text())
info.update({
    "status": "completed",
    "inputs": [str(Path(p).relative_to(PROJECT_ROOT)) for p in [injection_data[0], files["calibration_file"]]],
    "config_file": str(Path(config_file).resolve().relative_to(PROJECT_ROOT)),
    "parameters": {"background_index": 230, "trial_end_index": 500},
    "notes": "Preserved historical source calculation; not scientifically validated. See salt_dilution/README.md.",
})
info_path.write_text(json.dumps(info, indent=2) + "\n")
print("Saved run:", info_path.parent)
